Login to Hugging Face

In [1]:
from huggingface_hub import login
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

# Get token
hf_token = os.getenv("HF_TOKEN")

# Login
login(token=hf_token)

E:\Python\PDF-QA system\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Load EmbeddingGemma Model

In [2]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

MODEL_ID = "google/embeddinggemma-300M"

print("Loading EmbeddingGemma... (this may take a minute on first run)")
embedding_model = SentenceTransformer(MODEL_ID).to(device)

print(f"\nModel loaded on: {embedding_model.device}")
print(f"Embedding dimensions: {embedding_model.get_sentence_embedding_dimension()}")

Using device: cpu
Loading EmbeddingGemma... (this may take a minute on first run)

Model loaded on: cpu
Embedding dimensions: 768


C:\Users\Durvesh Narkhede\AppData\Local\Temp\ipykernel_14788\2572464382.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimensions: {embedding_model.get_sentence_embedding_dimension()}")


PDF -> JSON structure

In [14]:
from google import genai
from google.genai import types
import json

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

with open("data/notes.pdf", "rb") as f:
    pdf_bytes = f.read()

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        types.Part.from_bytes(
            data=pdf_bytes,
            mime_type="application/pdf",
        ),
        """
Role:
You are an expert Academic Transcription Assistant specializing in Software Project Management (SPM).

Objective:
Transcribe the attached handwritten PDF into structured Markdown AND structured JSON knowledge objects optimized for semantic retrieval and RAG systems.

Critical Rules:
- Do NOT summarize.
- Do NOT omit any content.
- Preserve full informational completeness.
- Correct obvious spelling errors while preserving original meaning.
- If handwriting is unclear, write: [Unclear Text: possible interpretation].
- Do NOT invent missing information.

Markdown Formatting:

# Headings
# Unit titles
## major sections
### sub-sections

Paragraph Structure:
- Keep related concepts together
- Maximum 6–8 lines
- Use bullet points where useful

Terminology:
Write full technical terms at least once.

Example:
Critical Path Method (CPM)
Program Evaluation and Review Technique (PERT)
Activity-on-Node (AON)
Precedence Network Diagram

Diagrams:
Create section:

### [Diagram Analysis]

Include:
- Components
- Logical flow
- Dependencies

Tables:
Recreate as Markdown tables.

Charts:
Explain axes and trends.

Concept Markers:
**Definition:**
**Process Steps:**
**Advantages:**
**Disadvantages:**

JSON Knowledge Extraction:

Each object format:

{
"id": "...",
"unit": "...",
"topic": "...",
"section": "...",
"type": "definition | process | explanation | diagram | table | advantages | disadvantages",
"keywords": [],
"content": "..."
}

Return ONLY JSON:

{
"markdown": "...",
"knowledge_objects": []
}
"""
    ],
    config=types.GenerateContentConfig(
        response_mime_type="application/json"
    )
)

data = json.loads(response.text)

markdown_text = data["markdown"]
knowledge_objects = data["knowledge_objects"]

In [17]:
with open("data/notes.md", "w", encoding="utf-8") as f:
    f.write(markdown_text)

In [20]:
with open("data/notes.json", "w", encoding="utf-8") as f:
    json.dump(knowledge_objects, f, indent=2, ensure_ascii=False)

Load JSON & Build LangChain Documents

In [3]:
from langchain_core.documents import Document
import json

# Load the JSON file saved earlier
with open("data/notes.json", "r", encoding="utf-8") as f:
    knowledge_objects = json.load(f)

print(f"Loaded {len(knowledge_objects)} knowledge objects")

# --- Embedding text builder ---
# We concatenate all metadata fields into a single string.
# EmbeddingGemma encodes this full string into one vector.
# More context = better semantic representation.
def build_embedding_text(obj):
    keywords = ", ".join(obj.get("keywords", []))
    text = f"""Unit: {obj.get('unit', '')}
Topic: {obj.get('topic', '')}
Section: {obj.get('section', '')}
Type: {obj.get('type', '')}
Keywords: {keywords}
{obj.get('content', '')}"""
    return text.strip()
    
# Why LangChain Document?
#   ChromaDB's LangChain wrapper expects Document objects.
#   The metadata dict is stored separately in Chroma alongside the vector,
#   so you can filter queries later (e.g., "only search Unit 3, type=definition").
# --- Convert to LangChain Documents ---
documents = []
for obj in knowledge_objects:
    text = build_embedding_text(obj)
    metadata = {
        "id": obj.get("id", ""),
        "unit": obj.get("unit", ""),
        "topic": obj.get("topic", ""),
        "section": obj.get("section", ""),
        "type": obj.get("type", "")
    }
    doc = Document(page_content=text, metadata=metadata)
    documents.append(doc)

print(f"Built {len(documents)} LangChain Documents")
print("\n--- Sample Document ---")
print(f"Content preview:\n{documents[0].page_content[:300]}...")
print(f"Metadata: {documents[0].metadata}")

Loaded 49 knowledge objects
Built 49 LangChain Documents

--- Sample Document ---
Content preview:
Unit: Unit-III
Topic: Activity Based Scheduling
Section: Introduction
Type: definition
Keywords: Activity Based Scheduling, project scheduling, identifying activities, organizing activities, sequencing activities
Activity Based Scheduling is a project scheduling approach that focuses on identifying,...
Metadata: {'id': 'scheduling-unit-iii-001', 'unit': 'Unit-III', 'topic': 'Activity Based Scheduling', 'section': 'Introduction', 'type': 'definition'}


Create a LangChain-Compatible EmbeddingGemma Wrapper

In [4]:
# Why a custom wrapper?
#   LangChain's Chroma integration requires an object that implements
#   the Embeddings base class with two methods:
#     - embed_documents(texts: List[str]) -> List[List[float]]
#     - embed_query(text: str) -> List[float]
#
#   There's no official LangChain package for EmbeddingGemma yet,
#   so we write a thin wrapper around our SentenceTransformer model.
#
# Why "Retrieval-document" prompt for documents and "Retrieval-query" for queries?
#   EmbeddingGemma was trained with task-specific instruction prefixes.
#   Using the correct prompt_name significantly improves retrieval accuracy
#   in RAG systems — it's the difference between ~0.6 and ~0.85 similarity scores.
#   See the EmbeddingGemma documentation you shared above.
#
# Why BATCH_SIZE=16 on CPU?
#   Larger batches = more RAM usage with no speed gain on CPU.
#   16 is a safe default. Reduce to 8 if you see memory warnings.

from langchain_core.embeddings import Embeddings
from tqdm import tqdm
from typing import List

BATCH_SIZE = 16  # Tune this down to 8 if you run into memory issues on CPU

class EmbeddingGemmaLangChain(Embeddings):
    """
    LangChain-compatible wrapper around SentenceTransformer (EmbeddingGemma).
    Uses task-specific prompts for documents vs. queries, as recommended
    in the EmbeddingGemma documentation for RAG pipelines.
    """

    def __init__(self, model: SentenceTransformer, batch_size: int = BATCH_SIZE):
        self.model = model
        self.batch_size = batch_size

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        """
        Encode a list of document texts.
        Uses prompt_name="Retrieval-document" — optimized for passages/docs.
        Shows a progress bar since CPU encoding can take time.
        """
        all_embeddings = []
        
        # Process in batches with a progress bar
        for i in tqdm(range(0, len(texts), self.batch_size), desc="Embedding documents"):
            batch = texts[i : i + self.batch_size]
            batch_embeddings = self.model.encode(
                batch,
                prompt_name="Retrieval-document",
                normalize_embeddings=True,   # Unit-length vectors → cosine sim = dot product
                show_progress_bar=False       # We use tqdm above instead
            )
            all_embeddings.extend(batch_embeddings.tolist())
        
        return all_embeddings

    def embed_query(self, text: str) -> List[float]:
        """
        Encode a single query string.
        Uses prompt_name="Retrieval-query" — optimized for user questions.
        This asymmetry (query vs doc prompts) is key to EmbeddingGemma's design.
        """
        embedding = self.model.encode(
            text,
            prompt_name="Retrieval-query",
            normalize_embeddings=True
        )
        return embedding.tolist()


# Instantiate the wrapper
embeddings = EmbeddingGemmaLangChain(model=embedding_model, batch_size=BATCH_SIZE)

# Quick sanity check — test both methods work before hitting Chroma
print("Testing embed_query...")
test_vec = embeddings.embed_query("What is the critical path method?")
print(f"  Query vector shape: {len(test_vec)} dimensions ✓")

print("Testing embed_documents with 2 samples...")
test_docs = embeddings.embed_documents([documents[0].page_content, documents[1].page_content])
print(f"  Document vectors: {len(test_docs)} vectors, each {len(test_docs[0])} dims ✓")

Testing embed_query...
  Query vector shape: 768 dimensions ✓
Testing embed_documents with 2 samples...


Embedding documents: 100%|███████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.16it/s]

  Document vectors: 2 vectors, each 768 dims ✓


Build ChromaDB Vector Store

In [27]:
# Why ChromaDB?
#   - Runs entirely locally — no cloud, no API key needed
#   - persist_directory means your vectors survive notebook restarts
#   - LangChain's Chroma wrapper handles all the batching/upserting for us
#
# What happens under the hood:
#   Chroma calls embeddings.embed_documents() on all document texts,
#   stores the resulting vectors alongside metadata and raw text,
#   then writes everything to ./chroma_langchain_db on disk.
#
# NOTE: On CPU with 100+ documents this will take several minutes.
#   The tqdm bar in embed_documents() will show you progress.
#   Don't interrupt — if it fails midway, delete ./chroma_langchain_db
#   and re-run this cell cleanly.

from langchain_chroma import Chroma

CHROMA_DIR = "./chroma_langchain_db"
COLLECTION_NAME = "spm_knowledge"

print(f"Starting embedding + ingestion of {len(documents)} documents...")
print(f"Collection: '{COLLECTION_NAME}'  |  Storage: '{CHROMA_DIR}'")
print("This will take a few minutes on CPU...\n")

vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DIR,
)

print(f"\nDone! Vector store built and persisted to '{CHROMA_DIR}'")
print(f"Total vectors stored: {vector_store._collection.count()}")

Starting embedding + ingestion of 49 documents...
Collection: 'spm_knowledge'  |  Storage: './chroma_langchain_db'
This will take a few minutes on CPU...



Embedding documents: 100%|███████████████████████████████████████████████████████████████| 4/4 [00:27<00:00,  6.90s/it]



✅ Done! Vector store built and persisted to './chroma_langchain_db'
Total vectors stored: 49


Reload Vector Store

In [5]:
# You don't need to re-embed everything each time.
# Just point Chroma to the same persist_directory and collection_name.
# The embedding function is still needed to embed new QUERIES at search time.

from langchain_chroma import Chroma
vector_store = Chroma(
    collection_name="spm_knowledge",
    embedding_function=embeddings,   
    persist_directory="./chroma_langchain_db",
)
print(f"Reloaded. Total vectors: {vector_store._collection.count()}")

Reloaded. Total vectors: 49


Verify with a Test Retrieval

In [29]:
TEST_QUERY = "Network planning model with diagram"
TOP_K = 3

print(f"Test query: \"{TEST_QUERY}\"\n")
results = vector_store.similarity_search_with_score(TEST_QUERY, k=TOP_K)

for rank, (doc, score) in enumerate(results, 1):
    print(f"--- Result {rank} | Similarity: {score:.4f} ---")
    print(f"Topic   : {doc.metadata.get('topic')}")
    print(f"Section : {doc.metadata.get('section')}")
    print(f"Type    : {doc.metadata.get('type')}")
    print(f"Preview : {doc.page_content[:200]}...")
    print()

Test query: "Network planning model with diagram"

--- Result 1 | Similarity: 0.6734 ---
Topic   : Network Planning Model
Section : Network Planning Model
Type    : diagram
Preview : Unit: Unit-III
Topic: Network Planning Model
Section: Network Planning Model
Type: diagram
Keywords: overall system, select module, check specifications, design module, code/test module, integrate/tes...

--- Result 2 | Similarity: 0.7896 ---
Topic   : Network Planning Model
Section : Network Planning Model
Type    : definition
Preview : Unit: Unit-III
Topic: Network Planning Model
Section: Network Planning Model
Type: definition
Keywords: project activities, interactions, network model, scheduling systems, time flow
In a Network Plan...

--- Result 3 | Similarity: 0.9006 ---
Topic   : Network Scheduling Techniques
Section : Network Planning Model
Type    : explanation
Preview : Unit: Unit-III
Topic: Network Scheduling Techniques
Section: Network Planning Model
Type: explanation
Keywords: network model, CP

Imports & Configuration

In [6]:
from typing import Optional
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage, AIMessage
from exa_py import Exa

# ── Configuration ──────────────────────────────────────────────────────────────
OLLAMA_MODEL      = "gemma4:e2b"
EXA_API_KEY       = os.getenv("EXA_API_KEY")   
CHROMA_DIR        = "./chroma_langchain_db"
COLLECTION_NAME   = "spm_knowledge"

# Gemma 4 sampling params
TEMPERATURE       = 1.0
TOP_P             = 0.95
TOP_K             = 64

# Retrieval settings
TOP_K_RETRIEVAL   = 4      # How many chunks to return per retrieval call

# Exa settings — highlights for agent workflows (best token efficiency)
EXA_NUM_RESULTS      = 3
EXA_MAX_CHARS        = 3000  

print("Configuration loaded")
print(f"   Model       : {OLLAMA_MODEL}")
print(f"   ChromaDB    : {CHROMA_DIR}")
print(f"   Retrieval K : {TOP_K_RETRIEVAL}")
print(f"   Exa results : {EXA_NUM_RESULTS}")

Configuration loaded
   Model       : gemma4:e2b
   ChromaDB    : ./chroma_langchain_db
   Retrieval K : 4
   Exa results : 3


Define Tool Functions

In [7]:
# Important design rule:
#   Always return a STRING from every tool.
#   LangChain's ToolMessage content must be a string.
#   We serialize results to JSON strings for consistency.

# ── Tool 1: retrieve_chunks ───────────────────────────────────────────────────
def retrieve_chunks(
    query: str,
    top_k: int = TOP_K_RETRIEVAL,
    filter_type: Optional[str] = None,
    filter_unit: Optional[str] = None
) -> str:
    """
    Search the PDF knowledge base (ChromaDB) for relevant chunks.
    Use this tool FIRST before web_search (web_search only if needed).
    Returns the most semantically similar chunks from the study notes.

    Args:
        query: The search query describing what information you need.
        top_k: Number of chunks to retrieve (default 4).
        filter_type: Optional. Filter by knowledge type.
                     (definition, process, explanation, diagram, table, advantages, disadvantages).
        filter_unit: Optional. Filter by unit name.
    """
    # Build Chroma metadata filter if requested
    where_filter = None
    if filter_type and filter_unit:
        where_filter = {
            "$and": [
                {"type": {"$eq": filter_type}},
                {"unit": {"$eq": filter_unit}}
            ]
        }
    elif filter_type:
        where_filter = {"type": {"$eq": filter_type}}
    elif filter_unit:
        where_filter = {"unit": {"$eq": filter_unit}}

    # Run similarity search
    results = vector_store.similarity_search_with_score(
        query,
        k=top_k,
        filter=where_filter
    )

    if not results:
        return json.dumps({"status": "no_results", "chunks": []})

    # Format results cleanly for the LLM
    chunks = []
    for doc, score in results:
        chunks.append({
            "similarity_score": round(float(score), 4),
            "topic": doc.metadata.get("topic", ""),
            "section": doc.metadata.get("section", ""),
            "type": doc.metadata.get("type", ""),
            "unit": doc.metadata.get("unit", ""),
            "content": doc.page_content
        })

    return json.dumps({
        "status": "success",
        "query": query,
        "num_results": len(chunks),
        "chunks": chunks
    }, ensure_ascii=False, indent=2)


# ── Tool 2: list_topics ───────────────────────────────────────────────────────
def list_topics() -> str:
    """
    List all topics, sections, units, and knowledge types
    available in the PDF knowledge base.
    Call this to understand what subjects are covered before
    answering a question, or when the user asks what topics are available.
    """
    # Pull all metadata from Chroma collection
    collection = vector_store._collection
    results = collection.get(include=["metadatas"])
    metadatas = results.get("metadatas", [])

    # Extract unique values per field
    topics   = sorted(set(m.get("topic",   "") for m in metadatas if m.get("topic")))
    sections = sorted(set(m.get("section", "") for m in metadatas if m.get("section")))
    units    = sorted(set(m.get("unit",    "") for m in metadatas if m.get("unit")))
    types    = sorted(set(m.get("type",    "") for m in metadatas if m.get("type")))

    return json.dumps({
        "status": "success",
        "total_chunks": len(metadatas),
        "units":    units,
        "topics":   topics,
        "sections": sections,
        "types":    types
    }, ensure_ascii=False, indent=2)


# ── Tool 3: web_search ────────────────────────────────────────────────────────
exa_client = Exa(api_key=EXA_API_KEY)

def web_search(
    query: str,
    num_results: int = EXA_NUM_RESULTS
) -> str:
    """
    Search the web using Exa for information NOT found in the PDF knowledge base.
    Only use this if retrieve_chunks did not return sufficient information.
    Returns highlights (key excerpts) from top web pages.

    Args:
        query: A clear, descriptive search query.
        num_results: Number of web results to return (default 3, max 5).
    """
    num_results = min(num_results, 5)

    try:
        response = exa_client.search_and_contents(
            query,
            type="auto",               
            num_results=num_results,
            contents={
                "highlights": {
                    "max_characters": EXA_MAX_CHARS  
                }
            }
        )
        
        results = []
        for r in response.results:
            results.append({
                "title":      r.title,
                "url":        r.url,
                "highlights": r.highlights if r.highlights else [],
                "published":  r.published_date
            })

        return json.dumps({
            "status":      "success",
            "query":       query,
            "num_results": len(results),
            "results":     results
        }, ensure_ascii=False, indent=2)

    except Exception as e:
        return json.dumps({
            "status": "error",
            "error":  str(e)
        })


print("Tools defined:")
print("   1. retrieve_chunks(query, top_k, filter_type, filter_unit)")
print("   2. list_topics()")
print("   3. web_search(query, num_results)")

Tools defined:
   1. retrieve_chunks(query, top_k, filter_type, filter_unit)
   2. list_topics()
   3. web_search(query, num_results)


Load Gemma 4 & Bind Tools

In [8]:
# .bind_tools() sends tool schemas to Ollama alongside every request.
# Ollama formats them correctly for Gemma 4's native function calling.
# When Gemma wants to call a tool, the response will have:
#   response.tool_calls = [{"name": "...", "args": {...}}]
# When Gemma has a final answer, tool_calls is empty and
#   response.content has the text.

llm = ChatOllama(
    model=OLLAMA_MODEL,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
)

# Bind all 3 tools — Ollama+LangChain auto-generates JSON schemas
# from the function signatures and docstrings
TOOLS = [retrieve_chunks, list_topics, web_search]

llm_with_tools = llm.bind_tools(TOOLS)

# Build a tool lookup map for the executor (Cell 5)
TOOL_MAP = {
    "retrieve_chunks": retrieve_chunks,
    "list_topics":     list_topics,
    "web_search":      web_search,
}

print(f"Gemma 4 E4B loaded and tools bound")
print(f"   Registered tools: {list(TOOL_MAP.keys())}")

Gemma 4 E4B loaded and tools bound
   Registered tools: ['retrieve_chunks', 'list_topics', 'web_search']


Tool Executor

In [9]:
# This function receives a tool_call from Gemma's response,
# looks up the Python function, calls it with the provided args,
# and returns a LangChain ToolMessage with the result.
#
# Why ToolMessage?
#   LangChain's message history needs a ToolMessage (not HumanMessage)
#   after a tool call, so Gemma knows it's reading a tool result
#   and not a new user question.
#
# Why validate the tool name?
#   Gemma could hallucinate a tool name that doesn't exist.
#   We catch that gracefully instead of crashing.

def execute_tool_call(tool_call: dict) -> ToolMessage:
    """
    Execute a single tool call from Gemma's response.
    Returns a ToolMessage to append to chat history.
    """
    tool_name = tool_call["name"]
    tool_args = tool_call["args"]
    tool_call_id = tool_call.get("id", tool_name)

    print(f"\n🔧 Tool called : {tool_name}")
    print(f"   Arguments   : {json.dumps(tool_args, indent=6)}")

    # Validate tool name exists
    if tool_name not in TOOL_MAP:
        result = json.dumps({
            "status": "error",
            "error": f"Unknown tool '{tool_name}'. Available: {list(TOOL_MAP.keys())}"
        })
    else:
        try:
            result = TOOL_MAP[tool_name](**tool_args)
        except TypeError as e:
            # Wrong arguments passed by Gemma
            result = json.dumps({
                "status": "error",
                "error": f"Invalid arguments for {tool_name}: {str(e)}"
            })
        except Exception as e:
            result = json.dumps({
                "status": "error",
                "error": f"Tool execution failed: {str(e)}"
            })

    # Parse result to show a clean preview in notebook output
    try:
        parsed = json.loads(result)
        status = parsed.get("status", "unknown")
        if tool_name == "retrieve_chunks":
            n = parsed.get("num_results", 0)
            print(f"   Result      : {status} — {n} chunks retrieved")
        elif tool_name == "list_topics":
            n = parsed.get("total_chunks", 0)
            t = len(parsed.get("topics", []))
            print(f"   Result      : {status} — {t} topics, {n} total chunks")
        elif tool_name == "web_search":
            n = parsed.get("num_results", 0)
            print(f"   Result      : {status} — {n} web results")
    except Exception:
        print(f"   Result      : returned")

    return ToolMessage(
        content=result,
        tool_call_id=tool_call_id
    )


print("Tool executor ready")

Tool executor ready


System Prompt Builder

In [10]:
# Why auto-load topics into the system prompt?
#   If Gemma already knows the scope of your notes at session start,
#   it can immediately retrieve relevant chunks without needing to
#   call list_topics first — saving one round trip.
#
# The system prompt also enforces the tool priority rule:
#   retrieve_chunks FIRST, web_search only as fallback.
#   Without this instruction, Gemma might prefer web search because
#   it's more "confident" about external knowledge.

def build_system_prompt() -> str:
    """
    Build the system prompt by auto-loading topics from ChromaDB.
    Called once at the start of each Q&A session.
    """
    print("Loading topics from ChromaDB for system prompt...")
    topics_json = list_topics()
    topics_data = json.loads(topics_json)

    units    = topics_data.get("units", [])
    topics   = topics_data.get("topics", [])
    types    = topics_data.get("types", [])
    n_chunks = topics_data.get("total_chunks", 0)

    units_str  = ", ".join(units)  if units  else "N/A"
    topics_str = "\n  - ".join(topics) if topics else "N/A"
    types_str  = ", ".join(types)  if types  else "N/A"

    system_prompt = f"""You are an expert academic assistant capable of helping across multiple subjects.
You have access to a structured knowledge base built from handwritten student notes (PDF).

## Knowledge Base Overview
- Total chunks indexed : {n_chunks}
- Units covered        : {units_str}
- Knowledge types      : {types_str}
- Topics available:
  - {topics_str}

## Tool Usage Rules — STRICTLY FOLLOW THIS ORDER
1. ALWAYS call `retrieve_chunks` first for any academic question.
2. Use the `filter_type` parameter when the question is clearly about
   a definition, process, advantages, disadvantages, or diagram.
3. Use the `filter_unit` parameter when the user specifies a unit.
4. Only call `web_search` if retrieve_chunks returns no useful results
   OR the user explicitly asks for external/recent information.
5. Call `list_topics` only if the user asks what topics are covered.

## Response Style
- Be concise and academically accurate.
- Structure your answers with clear headings when appropriate.
- If you used retrieve_chunks, base your answer ONLY on what was retrieved.
- If you used web_search, mention that the information came from the web.
- Never make up information that wasn't in the retrieved content.
"""
    print(f"System prompt built ({n_chunks} chunks, {len(topics)} topics)")
    return system_prompt


print("System prompt builder ready")

System prompt builder ready


The Agentic Loop

In [11]:
# This is the core engine. It runs until Gemma gives a final text answer.
#
# Loop steps:
#   1. Send current message history to Gemma
#   2. If Gemma returns tool_calls → execute each → append results → repeat
#   3. If Gemma returns plain text → that's the final answer → stop
#
# Why MAX_ITERATIONS?
#   Safety guard. If Gemma gets stuck in a tool-calling loop
#   (unlikely but possible), we stop and return what we have.
#
# Why append AIMessage before ToolMessages?
#   LangChain requires the assistant's tool_call message to appear
#   in history BEFORE the tool results. This mirrors how the model
#   "asked" for the tool and then "received" the answer.

MAX_ITERATIONS = 6   # Max tool calls per question before forcing final answer

def run_agent(user_question: str, chat_history: list, system_prompt: str) -> tuple[str, list]:
    """
    Run the agentic loop for one user question.
    
    Args:
        user_question : The user's question string
        chat_history  : Existing message history (list of LangChain messages)
        system_prompt : The system prompt string from build_system_prompt()
    
    Returns:
        (final_answer: str, updated_chat_history: list)
    """
    print(f"\n{'='*60}")
    print(f"❓ Question: {user_question}")
    print(f"{'='*60}")

    # Append new user question to history
    chat_history.append(HumanMessage(content=user_question))

    # Build full message list: system + history
    messages = [SystemMessage(content=system_prompt)] + chat_history

    for iteration in range(MAX_ITERATIONS):
        print(f"\n⟳ Iteration {iteration + 1}/{MAX_ITERATIONS}")

        # Send to Gemma 4 via Ollama
        response = llm_with_tools.invoke(messages)

        # ── Case 1: Gemma wants to call tools ─────────────────────────────────
        if response.tool_calls:
            print(f"   Gemma requested {len(response.tool_calls)} tool call(s)")

            # Append Gemma's tool-call message to history
            messages.append(response)
            chat_history.append(response)

            # Execute each tool call and collect results
            tool_messages = []
            for tool_call in response.tool_calls:
                tool_msg = execute_tool_call(tool_call)
                tool_messages.append(tool_msg)

            # Append all tool results to history
            messages.extend(tool_messages)
            chat_history.extend(tool_messages)

            # Continue loop — Gemma will now read the results
            continue

        # ── Case 2: Gemma has a final answer ──────────────────────────────────
        final_answer = response.content.strip()

        if final_answer:
            print(f"\nFinal answer received (iteration {iteration + 1})")
            # Append assistant's final answer to history
            chat_history.append(AIMessage(content=final_answer))
            return final_answer, chat_history

        # ── Case 3: Empty response (shouldn't happen, but handle it) ──────────
        print("Empty response from model, retrying...")
        messages.append(HumanMessage(content="Please provide your answer based on the retrieved information."))

    # Max iterations reached — return whatever the last content was
    fallback = "I was unable to generate a complete answer after multiple attempts. Please try rephrasing your question."
    chat_history.append(AIMessage(content=fallback))
    return fallback, chat_history


print("Agentic loop ready")

Agentic loop ready


Initialize Session

In [12]:
# Run this cell ONCE at the start of a session.
# It builds the system prompt (auto-loads topics) and
# creates a fresh chat history.
#
# To start a NEW session (clear memory), re-run this cell.
# To CONTINUE a session, just run next cell repeatedly.

system_prompt = build_system_prompt()
chat_history  = []   # Fresh session — no prior context

print("\nSession initialized — ready for questions!")
print("   Run next cell to ask your first question.")

Loading topics from ChromaDB for system prompt...
System prompt built (49 chunks, 42 topics)

Session initialized — ready for questions!
   Run next cell to ask your first question.


Ask a Question

In [13]:
# Change USER_QUESTION to whatever you want to ask.
# Re-run this cell for each new question in the same session.
# Chat history is preserved automatically across runs.
#
# Try these example questions:
#   "Explain Network Planning Model with diagram?"
#   "What are the advantages and disadvantages of PERT?"
#   "List all topics covered in my notes."
#   "What is the difference between CPM and PERT?"
#   "Search the web for latest project management tools in 2026."

USER_QUESTION = "Explain Network Planning Model with diagram?"

answer, chat_history = run_agent(
    user_question=USER_QUESTION,
    chat_history=chat_history,
    system_prompt=system_prompt
)

print(f"\n{'='*60}")
print("ANSWER")
print(f"{'='*60}")
print(answer)
print(f"\nSession history: {len(chat_history)} messages so far")


❓ Question: Explain Network Planning Model with diagram?

⟳ Iteration 1/6
   Gemma requested 1 tool call(s)

🔧 Tool called : retrieve_chunks
   Arguments   : {
      "query": "Network Planning Model with diagram"
}
   Result      : success — 4 chunks retrieved

⟳ Iteration 2/6

Final answer received (iteration 2)

ANSWER
The **Network Planning Model** is a fundamental concept in project scheduling systems where project activities and their interactions are modeled as networks to manage the flow of time and dependencies.

Based on the knowledge base, here is an explanation incorporating the context from the retrieved information:

### 1. Definition and Concept
The Network Planning Model essentially models:
*   **Project Activities:** The specific tasks that need to be performed.
*   **Interactions:** The relationships and dependencies between these activities.
*   **Time Flow:** How time progresses through these scheduled activities.

In this model, the goal is to structure the project